In [25]:
#!pip install skrebate scikit-learn pandas numpy matplotlib seaborn

$\textbf{Step 0: Importing all necessary libraries}$

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
import pickle
warnings.filterwarnings('ignore')

#Importing The ReliefF Library to Implement for Phase 2
from skrebate import ReliefF

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score

#Importing Libraries for the 5 Classifiers that will be used
from sklearn.svm import LinearSVC, SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

$\textbf{Step 1: Loading Datasets and Setting the Parameters for Models and}$
$\textbf{ReliefF Algorithm}$

In [27]:
DATASETS = {
    'Dataset1' : {'train': 'Datasets/1/train.csv','test': None},
    'Dataset2' : {'train': 'Datasets/2/train.csv','test': None},
    'Dataset3' : {'train': 'Datasets/3/train.csv','test': None},
    'Dataset4' : {'train': 'Datasets/4/train.csv','test': None},
    'Dataset5' : {'train': 'Datasets/5/train.csv','test': None},
    'Dataset6' : {'train': 'Datasets/6/train.csv','test': None},
    'Dataset7' : {'train': 'Datasets/7/train.csv','test': None},
    'Dataset8' : {'train': 'Datasets/8/train.csv','test': None},
}

LABEL_COL = 'Label'
N_FOLDS = 10
INNER_FOLDS = 3
RANDOM_STATE = 42

# ReliefF Settings
# k = number of top features to keep after ranking
# Try a few values; the best k is selected by inner CV
K_CANDIDATES = [5, 10, 20]  # adjust based on total feature count
RELIEFF_N_NEIGHBORS = 10          # ReliefF neighbourhood size


$\textbf{Step 2: Classifier Definition with Hyperparameter Grids}$

In [28]:
CLASSIFIERS = {
    'SVM': (
            CalibratedClassifierCV(LinearSVC(random_state=RANDOM_STATE, max_iter=2000)),
            {'classifier__estimator__C': [1, 10]}
            ),
    'kNN': (
        KNeighborsClassifier(),
        {
            'classifier__n_neighbors': [3, 5, 11, 21],
            'classifier__weights':     ['uniform', 'distance'],
        }
    ),
    'Decision Tree': (
        DecisionTreeClassifier(random_state=RANDOM_STATE),
        {
            'classifier__max_depth':        [5, 10, 20, None],
            'classifier__min_samples_split': [2, 5, 10],
        }
    ),
    'Random Forest': (
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        {
            'classifier__n_estimators': [50, 100, 200],
            'classifier__max_depth':    [5, 10, None],
        }
    ),
    'MLP': (
        MLPClassifier(max_iter=500, random_state=RANDOM_STATE, early_stopping=True),
        {
            'classifier__hidden_layer_sizes': [(64,), (128,)],
            'classifier__alpha':              [1e-4, 1e-3],
        }
    ),
}

$\textbf{Step 4: Helper Functions}$

In [29]:
def load_dataset(path, label_col=LABEL_COL):
    df = pd.read_csv(path)
    X = df.drop(columns=[label_col]).values.astype(np.float64) #Dropping the Label Column and converts to float64 Numpy array
    y_raw = df[label_col].values
    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    print(f'  Loaded: {X.shape[0]} samples, {X.shape[1]} features, '
          f'{len(np.unique(y))} classes')
    return X, y, le

def apply_relieff(X_train, y_train, X_test, k, n_neighbors=RELIEFF_N_NEIGHBORS, max_samples=500):
    # Subsample for ReliefF only — avoids distance matrix memory error
    if X_train.shape[0] > max_samples:
        idx = np.random.RandomState(RANDOM_STATE).choice(
            X_train.shape[0], max_samples, replace=False)
        X_relief = X_train[idx]
        y_relief = y_train[idx]
    else:
        X_relief = X_train
        y_relief = y_train

    imputer = SimpleImputer(strategy='mean')
    X_relief_imp = imputer.fit_transform(X_relief)

    selector = ReliefF(n_features_to_select=k, n_neighbors=n_neighbors)
    selector.fit(X_relief_imp, y_relief)
    top_idx = np.argsort(selector.feature_importances_)[::-1][:k]
    return X_train[:, top_idx], X_test[:, top_idx], selector.feature_importances_, top_idx

def build_pipeline(clf):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler()),
        ('classifier', clf),
    ])

def evaluate_classifier(clf, param_grid, X_train, y_train, X_test, y_test):
    pipe = build_pipeline(clf)
    inner_cv = StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True,
                                random_state = RANDOM_STATE)
    gs = GridSearchCV(pipe, param_grid, cv=inner_cv,
                      scoring='f1_macro', n_jobs=-1, refit=True)
    gs.fit(X_train, y_train)
    y_pred = gs.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division = 0)
    best_params = gs.best_params_
    return acc, f1, best_params

def print_fold_results(fold_idx, n_folds, ds_name, all_results, clf_names, best_k):
    print(f"\n  ── Fold {fold_idx + 1}/{n_folds} Results ({ds_name}) | best_k={best_k} ──")
    
    rows = []
    for phase_name, classifiers in all_results[ds_name].items():
        for clf_name, metrics in classifiers.items():
            if len(metrics['acc']) == 0:
                continue
            rows.append({
                'Phase':      'Baseline' if phase_name == 'baseline' else 'ReliefF',
                'Classifier': clf_name,
                'Accuracy':   f"{np.mean(metrics['acc']):.4f} ± {np.std(metrics['acc']):.4f}",
                'Macro-F1':   f"{np.mean(metrics['f1']):.4f} ± {np.std(metrics['f1']):.4f}",
                'Best Params': metrics['params'][-1],
            })
    
    df = pd.DataFrame(rows)
    pd.set_option('display.max_colwidth', 60)
    display(df)
    print()


$\textbf{Step 5: Main Experiment Loop}$

In [30]:
all_results = {}
relieff_scores = {}

outer_cv = StratifiedKFold(n_splits= N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

for ds_name, paths in DATASETS.items():
    print(f'\n=== Dataset: {ds_name} ===')
    X, y, le = load_dataset(paths['train'])

    all_results[ds_name] = {
        'baseline': {clf: {'acc': [], 'f1': [], 'params': []} for clf in CLASSIFIERS},
        'relieff':  {clf: {'acc': [], 'f1': [], 'params': []} for clf in CLASSIFIERS},
    }
    fold_feature_scores = []

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X,y)):
        print(f'  Fold {fold_idx + 1}/{N_FOLDS}', end=' ... ')
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

         # ── Phase 1: baseline (full features) ────────────────────────────────
        for clf_name, (clf, grid) in CLASSIFIERS.items():
            t0 = time.time()
            acc, f1, params = evaluate_classifier(
                clf, grid, X_train, y_train, X_test, y_test)
            all_results[ds_name]['baseline'][clf_name]['acc'].append(acc)
            all_results[ds_name]['baseline'][clf_name]['f1'].append(f1)
            all_results[ds_name]['baseline'][clf_name]['params'].append(params)
            print(f"    {clf_name}: {time.time()-t0:.1f}s")

        # ── Phase 2: ReliefF — fit on training fold ONLY ─────────────────────      
        print("    Starting ReliefF...", end=" ", flush=True)
        t0 = time.time()
        _, _, importances, sorted_idx = apply_relieff(
            X_train, y_train, X_test, k=1)
        print(f"done ({time.time()-t0:.1f}s)", flush=True)
        # Choose best k by slicing only — no extra ReliefF calls
        best_k, best_k_score = K_CANDIDATES[0], -np.inf
        for k_candidate in K_CANDIDATES:
            if k_candidate > X_train.shape[1]:
                continue
            top_idx = np.argsort(importances)[::-1][:k_candidate]
            Xtr_k, Xte_k = X_train[:, top_idx], X_test[:, top_idx]
            inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)
            scores = []
            for itr, ite in inner.split(Xtr_k, y_train):
                rf = RandomForestClassifier(n_estimators=50,
                                            random_state=RANDOM_STATE, n_jobs=-1)
                rf.fit(Xtr_k[itr], y_train[itr])
                scores.append(f1_score(y_train[ite],
                                       rf.predict(Xtr_k[ite]),
                                       average='macro', zero_division=0))
            if np.mean(scores) > best_k_score:
                best_k_score = np.mean(scores)
                best_k = k_candidate

        print(f"    best_k selected: {best_k}", flush=True)
            
        # Slice final features using best_k
        top_idx_final = np.argsort(importances)[::-1][:best_k]
        X_train_r = X_train[:, top_idx_final]
        X_test_r  = X_test[:, top_idx_final]
        fold_feature_scores.append(importances)

        for clf_name, (clf, grid) in CLASSIFIERS.items():
            t0 = time.time()
            acc, f1, params = evaluate_classifier(
                clf, grid, X_train_r, y_train, X_test_r, y_test)
            all_results[ds_name]['relieff'][clf_name]['acc'].append(acc)
            all_results[ds_name]['relieff'][clf_name]['f1'].append(f1)
            all_results[ds_name]['relieff'][clf_name]['params'].append(params)
            print(f"    {clf_name}: {time.time()-t0:.1f}s")

        print('done')
        print_fold_results(fold_idx, N_FOLDS, ds_name, all_results, clf_name, best_k)


    relieff_scores[ds_name] = np.mean(fold_feature_scores, axis=0)
        
# # Average ReliefF scores across folds for plotting
with open('results_checkpoint.pkl', 'wb') as f:
    pickle.dump({'all_results': all_results, 'relieff_scores': relieff_scores}, f)
print(f'  Checkpoint saved for {ds_name}.')


=== Dataset: Dataset1 ===
  Loaded: 9583 samples, 19 features, 4 classes
  Fold 1/10 ...     SVM: 4.5s
    kNN: 1.2s
    Decision Tree: 0.5s
    Random Forest: 6.0s
    MLP: 9.4s
    Starting ReliefF... done (0.8s)
    best_k selected: 5
    SVM: 0.6s
    kNN: 0.4s
    Decision Tree: 0.2s
    Random Forest: 4.6s
    MLP: 2.0s
done

  ── Fold 1/10 Results (Dataset1) | best_k=5 ──


,Phase,Classifier,Accuracy,Macro-F1,Best Params
0,Baseline,SVM,0.9864 ± 0.0000,0.7372 ± 0.0000,{'classifier__estimator__C': 10}
1,Baseline,kNN,0.9906 ± 0.0000,0.7368 ± 0.0000,"{'classifier__n_neighbors': 5, 'classifier__weights': 'd..."
2,Baseline,Decision Tree,1.0000 ± 0.0000,1.0000 ± 0.0000,"{'classifier__max_depth': 10, 'classifier__min_samples_s..."
3,Baseline,Random Forest,1.0000 ± 0.0000,1.0000 ± 0.0000,"{'classifier__max_depth': None, 'classifier__n_estimator..."
4,Baseline,MLP,0.9823 ± 0.0000,0.7329 ± 0.0000,"{'classifier__alpha': 0.0001, 'classifier__hidden_layer_..."
5,ReliefF,SVM,0.8467 ± 0.0000,0.4786 ± 0.0000,{'classifier__estimator__C': 10}
6,ReliefF,kNN,0.9990 ± 0.0000,0.9990 ± 0.0000,"{'classifier__n_neighbors': 3, 'classifier__weights': 'd..."
7,ReliefF,Decision Tree,1.0000 ± 0.0000,1.0000 ± 0.0000,"{'classifier__max_depth': 10, 'classifier__min_samples_s..."
8,ReliefF,Random Forest,1.0000 ± 0.0000,1.0000 ± 0.0000,"{'classifier__max_depth': 10, 'classifier__n_estimators'..."
9,ReliefF,MLP,0.8394 ± 0.0000,0.4419 ± 0.0000,"{'classifier__alpha': 0.0001, 'classifier__hidden_layer_..."



  Fold 2/10 ...     SVM: 1.6s
    kNN: 1.2s
    Decision Tree: 0.6s
    Random Forest: 6.1s
    MLP: 8.8s
    Starting ReliefF... done (0.8s)
    best_k selected: 5
    SVM: 0.5s
    kNN: 0.4s
    Decision Tree: 0.2s
    Random Forest: 4.5s
    MLP: 2.1s
done

  ── Fold 2/10 Results (Dataset1) | best_k=5 ──


,Phase,Classifier,Accuracy,Macro-F1,Best Params
0,Baseline,SVM,0.9870 ± 0.0005,0.7357 ± 0.0015,{'classifier__estimator__C': 10}
1,Baseline,kNN,0.9911 ± 0.0005,0.7356 ± 0.0012,"{'classifier__n_neighbors': 3, 'classifier__weights': 'd..."
2,Baseline,Decision Tree,1.0000 ± 0.0000,1.0000 ± 0.0000,"{'classifier__max_depth': 10, 'classifier__min_samples_s..."
3,Baseline,Random Forest,1.0000 ± 0.0000,1.0000 ± 0.0000,"{'classifier__max_depth': None, 'classifier__n_estimator..."
4,Baseline,MLP,0.9838 ± 0.0016,0.7326 ± 0.0004,"{'classifier__alpha': 0.001, 'classifier__hidden_layer_s..."
5,ReliefF,SVM,0.8337 ± 0.0130,0.3521 ± 0.1266,{'classifier__estimator__C': 1}
6,ReliefF,kNN,0.9984 ± 0.0005,0.9941 ± 0.0049,"{'classifier__n_neighbors': 3, 'classifier__weights': 'd..."
7,ReliefF,Decision Tree,1.0000 ± 0.0000,1.0000 ± 0.0000,"{'classifier__max_depth': 10, 'classifier__min_samples_s..."
8,ReliefF,Random Forest,1.0000 ± 0.0000,1.0000 ± 0.0000,"{'classifier__max_depth': 10, 'classifier__n_estimators'..."
9,ReliefF,MLP,0.8796 ± 0.0401,0.5405 ± 0.0986,"{'classifier__alpha': 0.0001, 'classifier__hidden_layer_..."



  Fold 3/10 ...     SVM: 1.5s
    kNN: 1.2s
    Decision Tree: 0.6s
    Random Forest: 6.4s
    MLP: 7.7s
    Starting ReliefF... done (0.7s)
    best_k selected: 5
    SVM: 0.8s
    kNN: 0.6s
    Decision Tree: 0.2s
    Random Forest: 5.2s
    MLP: 2.3s
done

  ── Fold 3/10 Results (Dataset1) | best_k=5 ──


...


  Fold 9/10 ...     SVM: 1.4s
    kNN: 1.3s
    Decision Tree: 0.6s
    Random Forest: 5.9s
    MLP: 9.5s
    Starting ReliefF... done (0.8s)
    best_k selected: 5
    SVM: 0.5s
    kNN: 0.4s
    Decision Tree: 0.2s
    Random Forest: 20.4s
    MLP: 2.5s
done

  ── Fold 9/10 Results (Dataset8) | best_k=5 ──


,Phase,Classifier,Accuracy,Macro-F1,Best Params
0,Baseline,SVM,0.9798 ± 0.0037,0.7800 ± 0.0996,{'classifier__estimator__C': 10}
1,Baseline,kNN,0.9854 ± 0.0042,0.8091 ± 0.1187,"{'classifier__n_neighbors': 5, 'classifier__weights': 'd..."
2,Baseline,Decision Tree,1.0000 ± 0.0000,1.0000 ± 0.0000,"{'classifier__max_depth': 10, 'classifier__min_samples_s..."
3,Baseline,Random Forest,0.9997 ± 0.0005,0.9997 ± 0.0005,"{'classifier__max_depth': None, 'classifier__n_estimator..."
4,Baseline,MLP,0.9762 ± 0.0060,0.7988 ± 0.1208,"{'classifier__alpha': 0.001, 'classifier__hidden_layer_s..."
5,ReliefF,SVM,0.8194 ± 0.0003,0.2420 ± 0.0312,{'classifier__estimator__C': 1}
6,ReliefF,kNN,0.9984 ± 0.0013,0.9983 ± 0.0016,"{'classifier__n_neighbors': 11, 'classifier__weights': '..."
7,ReliefF,Decision Tree,1.0000 ± 0.0000,1.0000 ± 0.0000,"{'classifier__max_depth': 10, 'classifier__min_samples_s..."
8,ReliefF,Random Forest,1.0000 ± 0.0000,1.0000 ± 0.0000,"{'classifier__max_depth': 10, 'classifier__n_estimators'..."
9,ReliefF,MLP,0.9137 ± 0.0073,0.6841 ± 0.0935,"{'classifier__alpha': 0.0001, 'classifier__hidden_layer_..."



  Fold 10/10 ...     SVM: 2.1s
    kNN: 1.1s
    Decision Tree: 0.6s
    Random Forest: 6.2s
    MLP: 8.7s
    Starting ReliefF... done (0.8s)
    best_k selected: 5
    SVM: 0.5s
    kNN: 0.4s
    Decision Tree: 0.2s
    Random Forest: 22.4s
    MLP: 5.6s
done

  ── Fold 10/10 Results (Dataset8) | best_k=5 ──


,Phase,Classifier,Accuracy,Macro-F1,Best Params
0,Baseline,SVM,0.9792 ± 0.0040,0.7744 ± 0.0960,{'classifier__estimator__C': 10}
1,Baseline,kNN,0.9848 ± 0.0044,0.8257 ± 0.1232,"{'classifier__n_neighbors': 3, 'classifier__weights': 'd..."
2,Baseline,Decision Tree,1.0000 ± 0.0000,1.0000 ± 0.0000,"{'classifier__max_depth': 10, 'classifier__min_samples_s..."
3,Baseline,Random Forest,0.9997 ± 0.0005,0.9997 ± 0.0005,"{'classifier__max_depth': None, 'classifier__n_estimator..."
4,Baseline,MLP,0.9758 ± 0.0058,0.8156 ± 0.1251,"{'classifier__alpha': 0.0001, 'classifier__hidden_layer_..."
5,ReliefF,SVM,0.8229 ± 0.0107,0.2662 ± 0.0784,{'classifier__estimator__C': 10}
6,ReliefF,kNN,0.9980 ± 0.0016,0.9975 ± 0.0028,"{'classifier__n_neighbors': 3, 'classifier__weights': 'd..."
7,ReliefF,Decision Tree,1.0000 ± 0.0000,1.0000 ± 0.0000,"{'classifier__max_depth': 10, 'classifier__min_samples_s..."
8,ReliefF,Random Forest,1.0000 ± 0.0000,1.0000 ± 0.0000,"{'classifier__max_depth': 10, 'classifier__n_estimators'..."
9,ReliefF,MLP,0.9198 ± 0.0193,0.6876 ± 0.0893,"{'classifier__alpha': 0.001, 'classifier__hidden_layer_s..."



  Checkpoint saved for Dataset8.
